# Step 3 — Register Agent Tools as UC Functions

Registers 6 Python-callable SQL functions in `abd_supplychain_dev.gold`.
Each function maps to one agent tool from the design doc.

| Function | Goal | Gold Table |
|---|---|---|
| `get_order_by_id` | Order Intelligence | `gold_order_summary` |
| `get_orders_by_customer` | Order Intelligence | `gold_order_summary` |
| `get_orders_by_category` | Order Intelligence | `gold_order_summary` |
| `get_shipment_by_order` | Shipment & Delivery | `gold_shipment_performance` |
| `get_carrier_performance` | Shipment & Delivery | `gold_shipment_performance` |
| `get_delayed_shipments` | Shipment & Delivery | `gold_shipment_performance` |

**Design rules applied:**
- All functions are read-only (SELECT only)
- All return a JSON string - not a table- so the LLM can read results naturally
- Descriptions are written for the LLM, not just for humans

In [0]:
%sql
-- Tool 1: get_order_by_id
-- Answers: "What is the status of order #X?" "Show me details for order 500042."
-- Returns a JSON object with full order detail for one specific order.
CREATE OR REPLACE FUNCTION abd_supplychain_dev.gold.get_order_by_id(order_id BIGINT)
RETURNS STRING
COMMENT 'Returns full details for a single order given its order_id. Use when the user asks about a specific order by number or ID. Do NOT use for listing all orders of a customer — use get_orders_by_customer for that. Returns: order_id, customer_id, order_date, order_status, is_fulfilled, product_category, sku, price_tier, warehouse_name, warehouse_location, quantity, line_revenue_usd.'
RETURN (
  SELECT TO_JSON(NAMED_STRUCT(
    'order_id',           g.order_id,
    'customer_id',        g.customer_id,
    'order_date',         CAST(g.order_date AS STRING),
    'order_status',       g.order_status,
    'is_fulfilled',       g.is_fulfilled,
    'product_category',   g.product_category,
    'sku',                g.sku,
    'price_tier',         g.price_tier,
    'warehouse_name',     g.warehouse_name,
    'warehouse_location', g.warehouse_location,
    'quantity',           g.quantity,
    'line_revenue_usd',   g.line_revenue_usd
  ))
  FROM abd_supplychain_dev.gold.gold_order_summary g
  WHERE g.order_id = get_order_by_id.order_id
  LIMIT 1
);

In [0]:
%sql
-- Tool 2: get_orders_by_customer
-- Answers: "Which orders does customer #1000 have?" "Show me all orders for customer 5042."
-- Returns a JSON array of up to 50 most recent orders for a customer.
CREATE OR REPLACE FUNCTION abd_supplychain_dev.gold.get_orders_by_customer(customer_id INT)
RETURNS STRING
COMMENT 'Returns up to 50 most recent orders for a specific customer_id, sorted by order_date descending. Use when the user asks about all orders belonging to a customer. Do NOT use for a single order — use get_order_by_id for that. Returns a JSON array where each element contains: order_id, order_date, order_status, is_fulfilled, product_category, sku, warehouse_name, quantity, line_revenue_usd.'
RETURN (
  SELECT TO_JSON(COLLECT_LIST(NAMED_STRUCT(
    'order_id',         g.order_id,
    'order_date',       CAST(g.order_date AS STRING),
    'order_status',     g.order_status,
    'is_fulfilled',     g.is_fulfilled,
    'product_category', g.product_category,
    'sku',              g.sku,
    'warehouse_name',   g.warehouse_name,
    'quantity',         g.quantity,
    'line_revenue_usd', g.line_revenue_usd
  )))
  FROM (
    SELECT *
    FROM abd_supplychain_dev.gold.gold_order_summary
    WHERE customer_id = get_orders_by_customer.customer_id
    ORDER BY order_date DESC
    LIMIT 50
  ) g
);

In [0]:
%sql
-- Tool 3: get_orders_by_category
-- Answers: "How is Electronics performing?" "What is the total revenue for Apparel this year?"
-- Returns an aggregate summary (not row-level) for a category within a date range.
CREATE OR REPLACE FUNCTION abd_supplychain_dev.gold.get_orders_by_category(
  category   STRING,
  start_date STRING,
  end_date   STRING
)
RETURNS STRING
COMMENT 'Returns aggregated order performance for a product category within a date range. Use for category revenue analysis or order volume questions. Valid categories: Electronics, Industrial, Apparel, Logistics Equipment. Dates must be YYYY-MM-DD format. Returns: category, start_date, end_date, total_orders, fulfilled_orders, total_quantity, total_revenue_usd, avg_order_revenue_usd.'
RETURN (
  SELECT TO_JSON(NAMED_STRUCT(
    'category',               get_orders_by_category.category,
    'start_date',             get_orders_by_category.start_date,
    'end_date',               get_orders_by_category.end_date,
    'total_orders',           COUNT(*),
    'fulfilled_orders',       CAST(SUM(CASE WHEN g.is_fulfilled THEN 1 ELSE 0 END) AS BIGINT),
    'total_quantity',         CAST(SUM(g.quantity) AS BIGINT),
    'total_revenue_usd',      ROUND(SUM(g.line_revenue_usd), 2),
    'avg_order_revenue_usd',  ROUND(AVG(g.line_revenue_usd), 2)
  ))
  FROM abd_supplychain_dev.gold.gold_order_summary g
  WHERE UPPER(TRIM(g.product_category)) = UPPER(TRIM(get_orders_by_category.category))
    AND g.order_date >= CAST(get_orders_by_category.start_date AS DATE)
    AND g.order_date <= CAST(get_orders_by_category.end_date AS DATE)
);

In [0]:
%sql
-- Tool 4: get_shipment_by_order
-- Answers: "What happened to the shipment for order 500000?" "Was order 500042 delivered on time?"
-- Returns shipment and delivery detail for a specific order.
CREATE OR REPLACE FUNCTION abd_supplychain_dev.gold.get_shipment_by_order(order_id INT)
RETURNS STRING
COMMENT 'Returns shipment and delivery details for a specific order_id. Use when the user asks what happened to a shipment, whether it arrived on time, about delivery notes, carrier used, or shipping cost for a specific order. Returns: shipment_id, order_id, carrier, ship_date, delivery_lead_days, is_on_time, shipment_cost_usd, cost_per_unit_usd, order_status, delivery_notes.'
RETURN (
  SELECT TO_JSON(NAMED_STRUCT(
    'shipment_id',          s.shipment_id,
    'order_id',             s.order_id,
    'carrier',              s.carrier,
    'ship_date',            CAST(s.ship_date AS STRING),
    'delivery_lead_days',   s.delivery_lead_days,
    'is_on_time',           s.is_on_time,
    'shipment_cost_usd',    s.shipment_cost_usd,
    'cost_per_unit_usd',    s.cost_per_unit_usd,
    'order_status',         s.order_status,
    'delivery_notes',       s.delivery_notes
  ))
  FROM abd_supplychain_dev.gold.gold_shipment_performance s
  WHERE s.order_id = get_shipment_by_order.order_id
  LIMIT 1
);

In [0]:
%sql
-- Tool 5: get_carrier_performance
-- Answers: "How is FedEx performing?" "What is the on-time rate for each carrier?"
-- Returns aggregated KPIs per carrier. Pass NULL or empty string to get all carriers.
CREATE OR REPLACE FUNCTION abd_supplychain_dev.gold.get_carrier_performance(carrier STRING)
RETURNS STRING
COMMENT 'Returns performance metrics for one or all carriers. Pass a carrier name for a specific carrier, or NULL/empty string to get all carriers. Supports partial name matching (e.g. fedex matches Fedex Supply). Known carriers: Fedex Supply, Ups Freight, Dhl Express, Amazon Logistics, Xpo Logistics. Returns a JSON array where each element contains: carrier, total_shipments, on_time_shipments, on_time_pct, avg_delivery_lead_days, avg_shipment_cost_usd.'
RETURN (
  SELECT TO_JSON(COLLECT_LIST(stats))
  FROM (
    SELECT NAMED_STRUCT(
      'carrier',                  s.carrier,
      'total_shipments',          COUNT(*),
      'on_time_shipments',        CAST(SUM(CASE WHEN s.is_on_time THEN 1 ELSE 0 END) AS BIGINT),
      'on_time_pct',              ROUND(AVG(CASE WHEN s.is_on_time THEN 100.0 ELSE 0.0 END), 1),
      'avg_delivery_lead_days',   ROUND(AVG(CAST(s.delivery_lead_days AS DOUBLE)), 1),
      'avg_shipment_cost_usd',    ROUND(AVG(s.shipment_cost_usd), 2)
    ) AS stats
    FROM abd_supplychain_dev.gold.gold_shipment_performance s
    WHERE s.carrier IS NOT NULL
      AND (
        get_carrier_performance.carrier IS NULL
        OR TRIM(get_carrier_performance.carrier) = ''
        OR LOWER(s.carrier) LIKE CONCAT('%', LOWER(TRIM(get_carrier_performance.carrier)), '%')
      )
    GROUP BY s.carrier
  )
);

In [0]:
%sql
-- Tool 6: get_delayed_shipments
-- Answers: "Which shipments are delayed?" "Show me the top 10 most delayed shipments."
-- Returns delayed shipments sorted by worst delay first. max_results controls how many to return.
CREATE OR REPLACE FUNCTION abd_supplychain_dev.gold.get_delayed_shipments(max_results INT)
RETURNS STRING
COMMENT 'Returns a list of delayed shipments (is_on_time = false) sorted by longest delay first. Use when the user asks which shipments are delayed, late, or behind schedule. Pass max_results to control how many are returned (e.g. 10, 20, 50). Returns a JSON array where each element contains: shipment_id, order_id, carrier, ship_date, delivery_lead_days, shipment_cost_usd, delivery_notes.'
RETURN (
  SELECT TO_JSON(COLLECT_LIST(NAMED_STRUCT(
    'shipment_id',        s.shipment_id,
    'order_id',           s.order_id,
    'carrier',            s.carrier,
    'ship_date',          CAST(s.ship_date AS STRING),
    'delivery_lead_days', s.delivery_lead_days,
    'shipment_cost_usd',  s.shipment_cost_usd,
    'delivery_notes',     s.delivery_notes
  )))
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (ORDER BY delivery_lead_days DESC) AS rn
    FROM abd_supplychain_dev.gold.gold_shipment_performance
    WHERE is_on_time = FALSE
      AND carrier IS NOT NULL
  ) s
  WHERE s.rn <= COALESCE(get_delayed_shipments.max_results, 20)
);

In [0]:
%sql
-- Test 1: look up a specific order
SELECT abd_supplychain_dev.gold.get_order_by_id(500000) AS result;

-- Test 2: all orders for a customer
-- SELECT abd_supplychain_dev.gold.get_orders_by_customer(1000) AS result;

-- Test 3: Electronics performance in 2025
-- SELECT abd_supplychain_dev.gold.get_orders_by_category('Electronics', '2025-01-01', '2025-12-31') AS result;

-- Test 4: shipment detail for a specific order
-- SELECT abd_supplychain_dev.gold.get_shipment_by_order(500000) AS result;

-- Test 5: all carrier performance metrics
-- SELECT abd_supplychain_dev.gold.get_carrier_performance(NULL) AS result;

-- Test 6: top 5 most delayed shipments
-- SELECT abd_supplychain_dev.gold.get_delayed_shipments(5) AS result;

In [0]:
SELECT abd_supplychain_dev.gold.get_orders_by_customer(1000) AS result;

In [0]:
SELECT abd_supplychain_dev.gold.get_orders_by_category('Electronics', '2025-01-01', '2025-12-31') AS result;

In [0]:
SELECT abd_supplychain_dev.gold.get_shipment_by_order(500000) AS result;

In [0]:
SELECT abd_supplychain_dev.gold.get_carrier_performance(NULL) AS result;

In [0]:
SELECT abd_supplychain_dev.gold.get_delayed_shipments(5) AS result;

In [0]:
%sql
-- Confirm all 6 functions are visible in the gold schema
SHOW FUNCTIONS IN abd_supplychain_dev.gold;

In [0]:
-- gold_shipment_performance.order_status was built before the order_status fix
-- and still contains '{VALUES[-1]}'. This MERGE syncs it from silver.
MERGE INTO abd_supplychain_dev.gold.gold_shipment_performance AS target
USING abd_supplychain_dev.silver.silver_fact_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET
  target.order_status = source.order_status;

-- Verify
SELECT order_status, COUNT(*) AS cnt
FROM abd_supplychain_dev.gold.gold_shipment_performance
GROUP BY order_status
ORDER BY cnt DESC;